# Explainable Cost-Sensitive Gradient Tree Boosting for Gallbladder Disease Prediction
### End-to-End Clinical Machine Learning & Dual-Tier TreeSHAP Explainability Pipeline
**Team Members:** Kantipudi Venkat Rohith, Gajjala Karthik Reddy, Harsha Lingampalli, Tarun Tej  
**Institution:** School of Computer Science and Engineering, Vellore Institute of Technology (VIT), Chennai  
**GitHub Repository:** [Explainable-Clinical-AI-Gallbladder-Prediction](https://github.com/Rohith-stack-creater/Explainable-Clinical-AI-Gallbladder-Prediction)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohith-stack-creater/Explainable-Clinical-AI-Gallbladder-Prediction/blob/main/Explainable_Gallbladder_AI_Pipeline.ipynb)

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install -q xgboost shap scikit-learn seaborn matplotlib pandas numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import xgboost as xgb
import shap
import warnings
warnings.filterwarnings('ignore')
print('Environment ready.')

## 2. Load CDC NHANES Gallbladder Cohort (N = 6,629)

In [ ]:
# Load from local directory or raw GitHub
csv_url = 'https://raw.githubusercontent.com/Rohith-stack-creater/Explainable-Clinical-AI-Gallbladder-Prediction/main/data/nhanes_gallbladder_cohort.csv'
try:
    df = pd.read_csv('../02_Dataset/nhanes_gallbladder_cohort.csv')
except Exception:
    try:
        df = pd.read_csv('nhanes_gallbladder_cohort.csv')
    except Exception:
        df = pd.read_csv(csv_url)

print('Cohort dimensions:', df.shape)
print('Target class distribution:\n', df['gallbladder_disease'].value_counts())

## 3. Stratified Partitioning & Cost-Sensitive Weight Calculation

In [ ]:
X = df.drop(columns=['respondent_id', 'gallbladder_disease'])
y = df['gallbladder_disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f'Training: {len(X_train)} samples | Holdout Test: {len(X_test)} samples')
print(f'Analytical cost-sensitive weight (scale_pos_weight): {scale_pos_weight:.3f}')

## 4. Train Proposed Tuned CS-XGBoost Classifier

In [ ]:
clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss'
)
clf.fit(X_train, y_train)
print('Model training complete.')

## 5. Performance Evaluation on Holdout Test Partition

In [ ]:
y_probs = clf.predict_proba(X_test)[:, 1]
y_preds = (y_probs >= 0.50).astype(int)
cm = confusion_matrix(y_test, y_preds)
tn, fp, fn, tp = cm.ravel()

print('Confusion Matrix: TN =', tn, 'FP =', fp, 'FN =', fn, 'TP =', tp)
print(f'Accuracy:    {(tn+tp)/len(y_test)*100:.2f}%')
print(f'Sensitivity: {tp/(tp+fn)*100:.2f}%')
print(f'Specificity: {tn/(tn+fp)*100:.2f}%')
print(f'ROC-AUC:     {roc_auc_score(y_test, y_probs):.3f}')
print(f'PR-AUC:      {average_precision_score(y_test, y_probs):.3f}')

## 6. Dual-Tier TreeSHAP Explainability (Global Beeswarm & Patient Waterfall)

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test)

# Global Beeswarm Summary
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('Global TreeSHAP Feature Attribution Beeswarm', fontsize=13)
plt.show()

# Patient-Level Local Waterfall
plt.figure(figsize=(9, 5))
shap.plots.waterfall(shap_values[0], show=False)
plt.title('Patient-Level Local TreeSHAP Waterfall Profile', fontsize=12)
plt.show()